# Juan Experiments

PatchCore Lite baseline orchestration. Implementation lives in `src/`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.config import load_config
from src.common.data import SpacepressoDataModule
from src.common.paths import resolve_path
from src.common.seed import set_seed
from src.common.submission import SubmissionWriter, validate_submission
from src.common.training import ExperimentRunner
from src.common.visualization import show_predictions
from src.methods import get_method_class


In [ ]:
config = load_config(ROOT / "configs/patchcore_lite/juan_baseline.yaml")
set_seed(config.get("seed", 42))

dm = SpacepressoDataModule(**config["data"])
train_good = dm.load_train_good()
train_anomalies = dm.load_train_anomalies()
test = dm.load_test()

print({"train_good": len(train_good), "train_anomalies": len(train_anomalies), "test": len(test)})


In [ ]:
if train_good and test:
    Method = get_method_class(config["method"]["name"])
    method = Method(config)
    runner = ExperimentRunner(method, config)
    runner.fit(train_good)
    predictions = runner.predict(test)
    show_predictions(test, predictions, n=3)
else:
    predictions = {}
    print("Dataset is empty or not found. Update config['data']['root'] before running the baseline.")


In [ ]:
if predictions:
    output_path = resolve_path(config["submission"]["output_path"], ROOT)
    SubmissionWriter(output_path).write(predictions)
    validate_submission(output_path, expected_shape=(config["data"]["image_size"], config["data"]["image_size"]))
    print(output_path)
